# Research AI Task 7: Titanic Survival Prediction using Logistic Regression from Scratch
**Dataset:** `titanic.csv`  
**Objective:** Build a robust Binary Logistic Regression classifier from scratch (using only `numpy`, `pandas`, and `matplotlib`) without relying on `scikit-learn`.

## 1. Load Dataset & Initial Inspection
In this cell, we import essential libraries and load the Titanic dataset. We inspect its dimensions and verify the column structure.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load Dataset
df = pd.read_csv('titanic.csv')
print(f"Dataset Loaded Successfully! Shape: {df.shape}")
print(df.head(3))

## 2. Feature Extraction & Data Cleaning
Real-world datasets contain missing values and categorical strings that machine learning models cannot process directly. Here we:
* Impute missing values in `Age` and `Fare` with their respective medians.
* Impute missing values in `Embarked` with the mode.
* Map `Sex` to binary values (`male`: 0, `female`: 1).
* One-hot encode categorical features like `Embarked`.

In [2]:
# 2. Feature Extraction & Cleaning
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Fare'] = df['Fare'].fillna(df['Fare'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
embarked_dummies = pd.get_dummies(df['Embarked'], prefix='Embarked', drop_first=True, dtype=int)
df = pd.concat([df, embarked_dummies], axis=1)

feature_cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare'] + list(embarked_dummies.columns)
X = df[feature_cols].to_numpy()
y = df['Survived'].to_numpy()

print(f"Feature Matrix Shape after preprocessing: {X.shape}")

## 3. Train-Test Split
We split the dataset into an **80% training set** (for model optimization) and a **20% test set** (for unbiased evaluation). A fixed random seed ensures reproducibility.

In [3]:
# 3. Train-Test Split (80% Train, 20% Test)
rng = np.random.default_rng(42)
indices = rng.permutation(len(X))
split = int(0.8 * len(X))

X_train, X_test = X[indices[:split]], X[indices[split:]]
y_train, y_test = y[indices[:split]], y[indices[split:]]

print(f"Training samples: {len(X_train)} | Test samples: {len(X_test)}")

## 4. Feature Scaling & Bias Intercept Addition
To prevent features with larger numeric ranges (like `Fare`) from dominating gradient descent, we standardize all features to zero mean and unit variance based on training statistics. We also append a column of ones for the model intercept.

In [4]:
# 4. Feature Scaling & Bias Term
mean = X_train.mean(axis=0)
std = X_train.std(axis=0, ddof=0)
std[std == 0] = 1.0

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

# Add bias intercept term
X_train = np.c_[np.ones(len(X_train)), X_train]
X_test = np.c_[np.ones(len(X_test)), X_test]
print(f"Scaled Training Feature Matrix Shape: {X_train.shape}")

## 5. Model Training (Logistic Regression from Scratch)
We implement binary logistic regression using the Sigmoid activation function and Binary Cross-Entropy loss. Optimization is performed using **Gradient Descent** over 2,000 iterations.

In [5]:
# 5. Model Training via Gradient Descent
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

lr = 0.05
iterations = 2000
m, n = X_train.shape
theta = np.zeros(n)
cost_history = []

for i in range(iterations):
    h = sigmoid(X_train @ theta)
    epsilon = 1e-15
    cost = -np.mean(y_train * np.log(h + epsilon) + (1 - y_train) * np.log(1 - h + epsilon))
    cost_history.append(cost)
    
    gradient = (X_train.T @ (h - y_train)) / m
    theta -= lr * gradient

print(f"Model Training Complete! Final Cost: {cost_history[-1]:.4f}")

## 6. Comprehensive Evaluation Metrics
We compute a complete suite of classification metrics including **Accuracy**, **Precision**, **Recall**, **Specificity**, **F1-Score**, and **ROC-AUC**, displaying them in a clean summary table.

In [6]:
# 6. Comprehensive Evaluation & Metrics Table
test_probs = sigmoid(X_test @ theta)
test_preds = (test_probs >= 0.5).astype(int)

tp = np.sum((y_test == 1) & (test_preds == 1))
tn = np.sum((y_test == 0) & (test_preds == 0))
fp = np.sum((y_test == 0) & (test_preds == 1))
fn = np.sum((y_test == 1) & (test_preds == 0))

acc = (tp + tn) / len(y_test)
prec = tp / (tp + fp) if (tp + fp) > 0 else 0
rec = tp / (tp + fn) if (tp + fn) > 0 else 0
spec = tn / (tn + fp) if (tn + fp) > 0 else 0
f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

# Compute ROC-AUC from scratch
sorted_idx = np.argsort(test_probs)[::-1]
y_sort = y_test[sorted_idx]
pos_count, neg_count = np.sum(y_test == 1), np.sum(y_test == 0)
rank_sum = np.sum(np.where(y_sort == 1)[0] + 1)
roc_auc = (rank_sum - pos_count * (pos_count + 1) / 2) / (pos_count * neg_count)

metrics_table = pd.DataFrame({
    "Evaluation Metric": ["Accuracy", "Precision", "Recall (Sensitivity)", "Specificity", "F1-Score", "ROC-AUC"],
    "Result Value": [f"{acc:.4f}", f"{prec:.4f}", f"{rec:.4f}", f"{spec:.4f}", f"{f1:.4f}", f"{roc_auc:.4f}"]
})

display(metrics_table)
print(f"\nConfusion Matrix Counts -> TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")

## 7. Advanced Side-by-Side Visualizations
To present a complete analysis for your task report, we generate **three professional side-by-side plots**: 
1. **Training Loss Curve** (Convergence over iterations)
2. **Confusion Matrix Heatmap** (Classification breakdown)
3. **ROC Curve** (Model separability performance)

In [7]:
# 7. Professional Side-by-Side Visualizations (Loss, Confusion Matrix, ROC Curve)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Training Loss Curve
axes[0].plot(cost_history, color='royalblue', lw=2)
axes[0].set_title("Training Loss Curve", fontsize=12, pad=12)
axes[0].set_xlabel("Iteration", fontsize=11, labelpad=8)
axes[0].set_ylabel("Binary Cross-Entropy Loss", fontsize=11, labelpad=8)
axes[0].grid(True, linestyle='--', alpha=0.5)

# Plot 2: Confusion Matrix Heatmap
cm = np.array([[tn, fp], [fn, tp]])
cax = axes[1].matshow(cm, cmap=plt.cm.Blues, alpha=0.7)
fig.colorbar(cax, ax=axes[1])
axes[1].set_title("Confusion Matrix Heatmap", fontsize=12, pad=12)
axes[1].set_xlabel("Predicted Label", fontsize=11, labelpad=8)
axes[1].set_ylabel("True Label", fontsize=11, labelpad=8)
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(['Died (0)', 'Survived (1)'])
axes[1].set_yticklabels(['Died (0)', 'Survived (1)'])

for (i, j), val in np.ndenumerate(cm):
    axes[1].text(j, i, f"{val}", ha='center', va='center', fontsize=14, fontweight='bold', color='black')

# Plot 3: ROC Curve
thresholds = np.linspace(0, 1, 100)
tpr_list, fpr_list = [], []
for th in thresholds:
    preds = (test_probs >= th).astype(int)
    t = np.sum((y_test == 1) & (preds == 1))
    f = np.sum((y_test == 0) & (preds == 1))
    tpr = t / pos_count if pos_count > 0 else 0
    fpr = f / neg_count if neg_count > 0 else 0
    tpr_list.append(tpr)
    fpr_list.append(fpr)

axes[2].plot(fpr_list, tpr_list, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
axes[2].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess')
axes[2].set_xlim([0.0, 1.0])
axes[2].set_ylim([0.0, 1.05])
axes[2].set_xlabel("False Positive Rate", fontsize=11, labelpad=8)
axes[2].set_ylabel("True Positive Rate", fontsize=11, labelpad=8)
axes[2].set_title("Receiver Operating Characteristic (ROC)", fontsize=12, pad=12)
axes[2].legend(loc="lower right")
axes[2].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()